In [1]:
import mlflow
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("credit-scoring")

<Experiment: artifact_location='/home/rapha/ai-engineer/credit-scoring-mlops/mlruns', creation_time=1773648058682, experiment_id='1', last_update_time=1773648058682, lifecycle_stage='active', name='credit-scoring', tags={}, workspace='default'>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Chargement des données prétraitées
df_train = pd.read_csv("../data/processed/train_processed.csv")

# Séparation features / cible
X = df_train.drop(columns=["TARGET", "SK_ID_CURR"])
y = df_train["TARGET"]

print(f"Features : {X.shape[1]} colonnes")
print(f"Échantillons : {X.shape[0]} lignes")
print(f"Distribution cible :\n{y.value_counts(normalize=True).round(3)}")


Features : 111 colonnes
Échantillons : 307511 lignes
Distribution cible :
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

# Sous-échantillon stratifié pour accélérer les tests (10% du dataset)
# Mettre MODE_TEST = False pour l'entraînement final sur tout le dataset
MODE_TEST = True

if MODE_TEST:
    _, X_sample, _, y_sample = train_test_split(
        X, y, test_size=0.1, stratify=y, random_state=42
    )
    print(f"Mode test — échantillon : {X_sample.shape[0]} lignes")
else:
    X_sample, y_sample = X, y
    print(f"Mode complet — {X_sample.shape[0]} lignes")

Mode test — échantillon : 30752 lignes


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Identification des types de colonnes
num_cols = X_sample.select_dtypes(include='number').columns.tolist()
cat_cols = X_sample.select_dtypes(include='object').columns.tolist()

# Preprocesseur commun à tous les modèles :
# - numériques   : imputation médiane + normalisation
# - catégorielles : imputation mode + one-hot encoding
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), cat_cols),
], remainder='drop')

print(f'Colonnes numériques   : {len(num_cols)}')
print(f'Colonnes catégorielles : {len(cat_cols)}')

Colonnes numériques   : 98
Colonnes catégorielles : 13


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Pipeline : preprocesseur commun + régression logistique
model_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

# Validation croisée stratifiée (5 folds)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

with mlflow.start_run(run_name='logistic_regression'):
    mlflow.log_param('model', 'LogisticRegression')
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_param('preprocessor', 'median_imputer+scaler | mode_imputer+ohe')
    mlflow.log_param('cv_folds', 5)

    scores_auc = cross_val_score(model_lr, X_sample, y_sample, cv=cv, scoring='roc_auc')

    mlflow.log_metric('auc_mean', round(scores_auc.mean(), 4))
    mlflow.log_metric('auc_std', round(scores_auc.std(), 4))

    print(f'AUC moyen : {scores_auc.mean():.4f} ± {scores_auc.std():.4f}')

AUC moyen : 0.7533 ± 0.0087


In [6]:
from sklearn.ensemble import RandomForestClassifier

# Pipeline : preprocesseur commun + Random Forest
model_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

with mlflow.start_run(run_name='random_forest'):
    mlflow.log_param('model', 'RandomForest')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_param('cv_folds', 5)

    scores_auc = cross_val_score(model_rf, X_sample, y_sample, cv=cv, scoring='roc_auc')

    mlflow.log_metric('auc_mean', round(scores_auc.mean(), 4))
    mlflow.log_metric('auc_std', round(scores_auc.std(), 4))

    print(f'AUC moyen : {scores_auc.mean():.4f} ± {scores_auc.std():.4f}')

AUC moyen : 0.7242 ± 0.0061


In [7]:
from lightgbm import LGBMClassifier

# Pipeline : preprocesseur commun + LightGBM
model_lgbm = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LGBMClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

with mlflow.start_run(run_name='lightgbm'):
    mlflow.log_param('model', 'LightGBM')
    mlflow.log_param('n_estimators', 200)
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_param('cv_folds', 5)

    scores_auc = cross_val_score(model_lgbm, X_sample, y_sample, cv=cv, scoring='roc_auc')

    mlflow.log_metric('auc_mean', round(scores_auc.mean(), 4))
    mlflow.log_metric('auc_std', round(scores_auc.std(), 4))

    print(f'AUC moyen : {scores_auc.mean():.4f} ± {scores_auc.std():.4f}')

/home/rapha/ai-engineer/credit-scoring-mlops/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/rapha/ai-engineer/credit-scoring-mlops/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/rapha/ai-engineer/credit-scoring-mlops/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/rapha/ai-engineer/credit-scoring-mlops/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC moyen : 0.7357 ± 0.0093


/home/rapha/ai-engineer/credit-scoring-mlops/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
